In [5]:
import numpy as np
from llm.tokenizer import CharTokenizer
from llm.nn import Embedding,Linear, cross_entropy, manual_seed


text = open("data/fables.txt", encoding="utf-8").read()
tok = CharTokenizer(text)
data = np.array(tok.encode(text))
V = tok.vocab_size


manual_seed(0)
emb = Embedding(V,2, std=1.0)
head = Linear(2, V, std=0.5)
rng = np.random.default_rng(0)
params = emb.parameters() + head.parameters()
lr = 50.0


for step in range(1500):
    i = rng.integers(0, len(data) - 1, size=1024)
    loss = cross_entropy(head(emb(data[i])), data[i+1])

    for p in params:
        p.grad =None

    loss.backward()
    for p in params:
        p.data -= 2.0 * p.grad
print(f"final loss", round(float(loss.data), 3))

final loss 2.58


In [5]:
import numpy as np
from llm.tokenizer import CharTokenizer
from llm.nn import Embedding, manual_seed
tok = CharTokenizer(open("data/fables.txt", encoding="utf-8").read())
manual_seed(1)
tok_emb = Embedding(tok.vocab_size, 4, std=1.0)
# 4 numbers per token
pos_emb = Embedding(5, 4, std=1.0)
# positions 0..4
ids = np.array(tok.encode("a a a"))
x = tok_emb(ids) + pos_emb(np.arange(5))
print(np.round(tok_emb(ids).data, 1))
print(np.round(x.data, 1))

[[-0.3 -1.   0.  -1.1]
 [ 0.9  0.4 -0.5  0.6]
 [-0.3 -1.   0.  -1.1]
 [ 0.9  0.4 -0.5  0.6]
 [-0.3 -1.   0.  -1.1]]
[[-0.2 -1.1  0.2 -2.7]
 [ 2.7 -0.2 -2.1  1.2]
 [-0.7 -0.6 -0.3 -1.2]
 [ 1.2 -0.3  0.1  0.1]
 [-1.2 -0.9  0.5 -1.4]]


In [6]:
batch = np.array([tok.encode("The f"), tok.encode("A dog")])
x = tok_emb(batch) + pos_emb(np.arange(5))
print(tok_emb(batch).shape, pos_emb(np.arange(5)).shape, x.shape)

(2, 5, 4) (5, 4) (2, 5, 4)
